# <span style="color: #1F1DB5;">SPRINT 12 – ML en Negocios

# <span style="color: #1F1DB5;">Notebook 1 – Bootstrapping, Data Leakage y Pipelines Profesionales — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Comprender y aplicar bootstrapping para estimación de intervalos de confianza.
- Identificar y prevenir data leakage en pipelines de ML.
- Construir sklearn Pipelines completas para evitar leakage.
- Diferenciar entre enfoques Data-Centric vs Model-Centric en IA.
- Aplicar validación correcta con pipelines encadenados.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender y aplicar **bootstrapping** para estimación de intervalos de confianza.
- Identificar y prevenir **data leakage** en pipelines de ML.
- Construir **sklearn Pipelines** completas para evitar leakage.
- Diferenciar entre enfoques **Data-Centric vs Model-Centric** en IA.
- Aplicar validación correcta con **pipelines encadenados**.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Bootstrapping: muestreo con reemplazo | 20 min |
Data Leakage: detectar y prevenir | 25 min |
Pipelines de sklearn | 25 min |
Data-Centric vs Model-Centric AI | 10 min |
Actividad: Exposición por Equipos | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo podemos evitar que nuestro modelo "haga trampa" durante el entrenamiento?

Imagina que entrenas un modelo que te da 98% de accuracy en validación, pero al desplegarlo en producción su rendimiento cae al 70%. ¿Qué pasó? Probablemente tenías **data leakage** — información del futuro o del test que se filtró al entrenamiento.

Hoy aprenderemos a:
1. Medir la incertidumbre real de nuestros modelos (bootstrapping)
2. Detectar y prevenir fugas de información (data leakage)
3. Construir pipelines que garanticen integridad en el flujo de datos

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Bootstrapping: muestreo con reemplazo (20 mins)

### <span style="color: #2772F2;">¿Qué es bootstrapping?

Bootstrapping es una técnica estadística que nos permite **estimar la distribución de una estadística** (como la media, mediana, o accuracy de un modelo) usando **muestreo con reemplazo**.

**La idea clave:** Si solo tenemos una muestra de datos, podemos simular "muchas muestras" tomando subconjuntos aleatorios con reemplazo de nuestra muestra original. Cada subconjunto es una "muestra bootstrap".

### <span style="color: #2772F2;">¿Por qué lo necesitamos en ML?

Cuando reportamos "mi modelo tiene 85% de accuracy", esa es una **estimación puntual**. Pero ¿qué tan seguro estás de ese número? ¿Podría ser 83% o 87% con diferentes datos?

Bootstrapping nos da un **intervalo de confianza**: "mi modelo tiene entre 83% y 87% de accuracy con 95% de confianza". Esto es mucho más informativo para tomar decisiones de negocio.

### <span style="color: #2772F2;">Algoritmo:
1. Tomar N muestras con reemplazo del dataset original (mismo tamaño)
2. Calcular la estadística de interés en cada muestra
3. Repetir B veces (típicamente B = 1000 o más)
4. Los percentiles 2.5% y 97.5% de la distribución dan el intervalo de confianza al 95%

Implementemos bootstrapping para calcular el intervalo de confianza de la accuracy de un modelo de clasificación.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# Crear dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=6,
                           random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelo
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Accuracy puntual
y_pred = rf.predict(X_test)
acc_puntual = accuracy_score(y_test, y_pred)
print(f"Accuracy puntual: {acc_puntual:.4f}")
print("Pero... ¿qué tan confiable es este número?")

In [ ]:
# Bootstrapping para intervalo de confianza
np.random.seed(42)
n_bootstrap = 1000
n_test = len(y_test)

bootstrap_accuracies = []

for i in range(n_bootstrap):
    # Muestrear con reemplazo del conjunto de test
    indices = np.random.choice(n_test, size=n_test, replace=True)
    y_test_boot = y_test[indices]
    y_pred_boot = y_pred[indices]
    
    # Calcular accuracy en esta muestra bootstrap
    acc_boot = accuracy_score(y_test_boot, y_pred_boot)
    bootstrap_accuracies.append(acc_boot)

bootstrap_accuracies = np.array(bootstrap_accuracies)

# Intervalo de confianza al 95%
ci_lower = np.percentile(bootstrap_accuracies, 2.5)
ci_upper = np.percentile(bootstrap_accuracies, 97.5)

print(f"=== Intervalo de Confianza Bootstrap (95%) ===")
print(f"  Accuracy puntual: {acc_puntual:.4f}")
print(f"  IC 95%: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  Amplitud del intervalo: {ci_upper - ci_lower:.4f}")
print(f"\n  Interpretación: Con 95% de confianza, la accuracy real")
print(f"  del modelo está entre {ci_lower:.4f} y {ci_upper:.4f}")

In [ ]:
# Visualizar la distribución bootstrap
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(bootstrap_accuracies, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(acc_puntual, color='red', linestyle='--', linewidth=2, label=f'Accuracy puntual: {acc_puntual:.4f}')
ax.axvline(ci_lower, color='orange', linestyle=':', linewidth=2, label=f'IC inferior: {ci_lower:.4f}')
ax.axvline(ci_upper, color='orange', linestyle=':', linewidth=2, label=f'IC superior: {ci_upper:.4f}')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución Bootstrap de Accuracy (1000 iteraciones)')
ax.legend()
plt.tight_layout()
plt.show()

Preguntas:

- ¿Por qué el muestreo es "con reemplazo"? ¿Qué pasaría si fuera sin reemplazo?

- ¿Cómo cambia el ancho del intervalo si aumentamos el número de muestras bootstrap?

- ¿En qué situación de negocio sería crítico reportar intervalos de confianza en vez de solo un número?

### Mis respuestas de validación

- 
- 


### <span style="color: #2772F2;">Bootstrapping para comparar dos modelos

Bootstrapping también nos permite responder preguntas como: "¿Es el modelo A realmente mejor que el modelo B, o la diferencia es solo ruido estadístico?"

Si el intervalo de confianza de la diferencia incluye el 0, no podemos concluir que un modelo es superior al otro.

In [ ]:
# Comparar dos modelos con bootstrapping
from sklearn.tree import DecisionTreeClassifier

# Segundo modelo
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Bootstrap de la diferencia
bootstrap_diffs = []
for i in range(1000):
    indices = np.random.choice(n_test, size=n_test, replace=True)
    acc_rf_boot = accuracy_score(y_test[indices], y_pred[indices])
    acc_dt_boot = accuracy_score(y_test[indices], y_pred_dt[indices])
    bootstrap_diffs.append(acc_rf_boot - acc_dt_boot)

bootstrap_diffs = np.array(bootstrap_diffs)
ci_diff_lower = np.percentile(bootstrap_diffs, 2.5)
ci_diff_upper = np.percentile(bootstrap_diffs, 97.5)

print("=== ¿Es Random Forest mejor que Decision Tree? ===")
print(f"  Accuracy RF: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Accuracy DT: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"  Diferencia puntual: {accuracy_score(y_test, y_pred) - accuracy_score(y_test, y_pred_dt):.4f}")
print(f"\n  IC 95% de la diferencia: [{ci_diff_lower:.4f}, {ci_diff_upper:.4f}]")
if ci_diff_lower > 0:
    print("  ✅ RF es significativamente mejor (el IC no incluye 0)")
else:
    print("  ⚠️  No podemos concluir que RF sea mejor (el IC incluye 0)")

## <span style="color: #2826DE;">Data Leakage: detectar y prevenir (25 mins)

### <span style="color: #2772F2;">¿Qué es Data Leakage?

Data leakage (fuga de datos) ocurre cuando **información que no estaría disponible al momento de hacer una predicción** se filtra al proceso de entrenamiento. El modelo aprende "atajos" que no existirán en producción.

### <span style="color: #2772F2;">Tipos de Data Leakage:

**1. Target Leakage (fuga del target):**
Cuando un feature contiene información que se genera DESPUÉS del evento que estás prediciendo.
- Ejemplo: Predecir si un paciente tiene diabetes usando "dosis_insulina" como feature
- La dosis de insulina se prescribe DESPUÉS del diagnóstico — no la conocerías antes

**2. Train-Test Contamination:**
Cuando información del test set se filtra al train set durante el preprocesamiento.
- Ejemplo: Escalar datos ANTES de hacer el split → la media y std del test influyen en el train
- Ejemplo: Aplicar SMOTE a todo el dataset y luego hacer split

**3. Temporal Leakage:**
En datos de series temporales, usar información del futuro para predecir el pasado.
- Ejemplo: Predecir ventas del lunes usando datos del martes
- Ejemplo: Features calculados con ventanas que incluyen datos futuros

### <span style="color: #2772F2;">¿Cómo detectarlo?

🚩 **Red flags:**
- Accuracy sospechosamente alta (>98% en problemas difíciles)
- Rendimiento cae drásticamente al pasar a producción
- Un feature tiene correlación casi perfecta con el target
- Los resultados son "demasiado buenos para ser verdad"

Vamos a demostrar cómo el data leakage infla artificialmente las métricas y cómo prevenirlo.

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- DEMOSTRACIÓN DE DATA LEAKAGE ---
# Caso 1: Escalar ANTES del split (MAL)
X_full, y_full = make_classification(n_samples=500, n_features=5, random_state=42)

# INCORRECTO: escalar todo junto antes del split
scaler_mal = StandardScaler()
X_scaled_mal = scaler_mal.fit_transform(X_full)  # ⚠️ fit en TODO el dataset
X_train_mal, X_test_mal, y_train_l, y_test_l = train_test_split(
    X_scaled_mal, y_full, test_size=0.3, random_state=42)

rf_leaky = RandomForestClassifier(n_estimators=50, random_state=42)
rf_leaky.fit(X_train_mal, y_train_l)
acc_leaky = rf_leaky.score(X_test_mal, y_test_l)

# CORRECTO: escalar después del split, fit solo en train
X_train_ok, X_test_ok, y_train_ok, y_test_ok = train_test_split(
    X_full, y_full, test_size=0.3, random_state=42)

scaler_bien = StandardScaler()
X_train_scaled = scaler_bien.fit_transform(X_train_ok)  # ✅ fit solo en train
X_test_scaled = scaler_bien.transform(X_test_ok)        # ✅ transform en test

rf_clean = RandomForestClassifier(n_estimators=50, random_state=42)
rf_clean.fit(X_train_scaled, y_train_ok)
acc_clean = rf_clean.score(X_test_scaled, y_test_ok)

print("=== Demostración de Data Leakage ===")
print(f"\n  ⚠️  Con leakage (escalar antes del split): {acc_leaky:.4f}")
print(f"  ✅ Sin leakage (escalar después del split): {acc_clean:.4f}")
print(f"\n  Diferencia: {(acc_leaky - acc_clean)*100:.2f} puntos porcentuales")
print("\n  La diferencia puede parecer pequeña aquí, pero en datasets")
print("  reales con más preprocesamiento, puede ser enorme.")

Preguntas:

- ¿Por qué hacer fit_transform en todo el dataset antes del split causa leakage?

- ¿Puedes pensar en un ejemplo de target leakage en tu proyecto o en la vida real?

- ¿Qué herramienta de sklearn nos ayuda a prevenir este tipo de errores automáticamente?

### Mis respuestas de validación

- 
- 


### <span style="color: #2772F2;">Ejemplo práctico: Temporal Leakage

En datos temporales, el leakage temporal es especialmente peligroso. Imagina predecir ventas del lunes usando features que incluyen información del martes.

In [ ]:
# Ejemplo de temporal leakage
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=100, freq='D')
sales = np.cumsum(np.random.randn(100)) + 50  # ventas con tendencia

df_temporal = pd.DataFrame({'fecha': dates, 'ventas': sales})
df_temporal['ventas_dia_siguiente'] = df_temporal['ventas'].shift(-1)  # ⚠️ INFO DEL FUTURO
df_temporal['media_movil_7d'] = df_temporal['ventas'].rolling(7).mean()

print("=== Temporal Leakage ===")
print(df_temporal[['fecha', 'ventas', 'ventas_dia_siguiente', 'media_movil_7d']].head(10))
print("\n⚠️ 'ventas_dia_siguiente' es leakage: usa info del futuro")
print("✅ 'media_movil_7d' es válida: solo usa datos pasados")
print("\nRegla: en series temporales, los features solo pueden")
print("   usar información ANTERIOR al punto de predicción.")

## <span style="color: #2826DE;">Pipelines de sklearn (25 mins)

### <span style="color: #2772F2;">¿Por qué usar Pipeline?

El `Pipeline` de sklearn resuelve el problema del data leakage de forma elegante: **encapsula todo el flujo** (preprocesamiento + modelo) en un solo objeto que garantiza que:

1. **fit** se ejecuta SOLO en datos de entrenamiento
2. **transform** se aplica correctamente en train y test
3. No hay posibilidad de contaminar accidentalmente el test set
4. El código es más limpio, reproducible y desplegable

### <span style="color: #2772F2;">Pipeline + ColumnTransformer

La combinación más poderosa: Pipeline con ColumnTransformer permite aplicar diferentes transformaciones a diferentes tipos de columnas, todo de forma segura y en un solo objeto.

```python
Pipeline([
    ('preprocessing', ColumnTransformer([...])),
    ('model', RandomForestClassifier())
])
```

Cuando haces `pipeline.fit(X_train, y_train)`:
- El ColumnTransformer se ajusta (fit) solo con X_train
- El modelo se entrena con los datos transformados

Cuando haces `pipeline.predict(X_test)`:
- El ColumnTransformer aplica transform (NO fit) con X_test
- El modelo predice con datos transformados correctamente

Construyamos un pipeline completo que demuestra cómo previene el data leakage automáticamente.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

# Crear dataset más realista con valores faltantes
np.random.seed(42)
n = 600
df_pipe = pd.DataFrame({
    'edad': np.where(np.random.random(n) > 0.9, np.nan, np.random.randint(20, 60, n)),
    'ingreso': np.random.exponential(35000, n) + 15000,
    'antiguedad': np.random.exponential(5, n),
    'region': np.random.choice(['norte', 'sur', 'centro', 'este'], n),
    'tipo_contrato': np.random.choice(['temporal', 'indefinido', 'freelance'], n),
})
y_pipe = (df_pipe['ingreso'] > 40000).astype(int)

# Definir tipos de columnas
num_features = ['edad', 'ingreso', 'antiguedad']
cat_features = ['region', 'tipo_contrato']

# Pipeline de preprocesamiento numérico
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline de preprocesamiento categórico
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

# Combinar en ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_features),
    ('cat', categorical_pipeline, cat_features)
])

# Pipeline completo
full_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Cross-validation SEGURA: el pipeline garantiza no-leakage
scores = cross_val_score(full_pipeline, df_pipe, y_pipe, cv=5, scoring='f1')
print("=== Pipeline Completo con Cross-Validation ===")
print(f"  F1-score: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"\n  ✅ Sin data leakage: cada fold se preprocesa independientemente")
print(f"\nEstructura del pipeline:")
print(full_pipeline)

Preguntas:

- ¿Qué pasaría si hiciéramos el impute de NaN ANTES del cross_val_score?

- ¿Por qué el Pipeline garantiza que no hay leakage en cross-validation?

### Mis respuestas de validación

- 
- 


### <span style="color: #2772F2;">Tips y buenas prácticas con Pipelines

- **Siempre** usa Pipeline cuando tengas preprocesamiento + modelo.
- **Nunca** hagas fit_transform antes del split o del cross_val_score.
- Guarda el Pipeline completo con joblib, no solo el modelo.
- Usa `pipeline.named_steps` para acceder a partes individuales.
- En producción, un Pipeline es un solo objeto que recibe datos crudos y devuelve predicciones.

In [ ]:
# Acceder a partes del pipeline
print("=== Acceder a componentes del Pipeline ===\n")
full_pipeline.fit(df_pipe, y_pipe)

# Ver los pasos
print("Pasos del pipeline:")
for name, step in full_pipeline.named_steps.items():
    print(f"  '{name}': {type(step).__name__}")

# Acceder al modelo entrenado
model_inside = full_pipeline.named_steps['model']
print(f"\nModelo interno: {type(model_inside).__name__}")
print(f"  n_estimators: {model_inside.n_estimators}")
print(f"  Feature importances disponibles: {hasattr(model_inside, 'feature_importances_')}")

### <span style="color: #2772F2;">Resumen: ¿Cómo evitar data leakage?

| Tipo de Leakage | Ejemplo | Cómo prevenirlo |
|---|---|---|
Target Leakage | Feature derivado del target | Revisar causalidad temporal |
Train-Test Contamination | fit_transform en todo | Usar Pipeline de sklearn |
Temporal Leakage | Media móvil con datos futuros | Split temporal estricto |

**Regla de oro:** Pregúntate siempre — "¿tendría este dato disponible AL MOMENTO de hacer la predicción en producción?"

## <span style="color: #2826DE;">Data-Centric vs Model-Centric AI (10 mins)

### <span style="color: #2772F2;">El paradigma tradicional: Model-Centric

Históricamente, la comunidad de ML se ha enfocado en **mejorar los modelos**:
- Probar más algoritmos
- Hacer hyperparameter tuning exhaustivo
- Crear arquitecturas más complejas
- Más capas, más neuronas, más árboles

### <span style="color: #2772F2;">El nuevo paradigma: Data-Centric AI

Andrew Ng propuso un cambio de enfoque: en vez de mejorar el modelo, **mejorar los datos**:
- Limpiar etiquetas inconsistentes
- Mejorar la calidad de los features
- Aumentar datos en las zonas donde el modelo falla
- Curar datasets de forma sistemática

**¿Por qué funciona?**
- Un modelo simple con datos excelentes suele superar a un modelo complejo con datos mediocres.
- En producción, es más fácil mejorar la calidad de datos que redesañar un modelo.
- Los errores de datos (etiquetas incorrectas, ruido, inconsistencias) ponen un techo al rendimiento de cualquier modelo.

**Estrategias Data-Centric:**
1. **Data Cleaning**: Identificar y corregir etiquetas ruidosas
2. **Data Augmentation**: Generar más datos donde el modelo falla
3. **Feature Engineering**: Crear representaciones más informativas
4. **Consistency**: Asegurar que los criterios de etiquetado sean uniformes
5. **Slicing**: Analizar rendimiento por subgrupos, no solo globalmente

## <span style="color: #2826DE;">Tips y buenas prácticas

- 🔄 **Siempre usa Pipeline** para cualquier preprocesamiento + modelo.
- 📊 **Reporta intervalos de confianza** al presentar resultados a stakeholders.
- 🚫 **Desconfía de resultados muy buenos**: si parece demasiado bueno para ser verdad, probablemente es leakage.
- 🧪 **Valida en producción**: monitorea si el rendimiento en producción es similar al de validación.
- 📅 **En datos temporales**: siempre haz split respetando el orden cronológico.
- 🔧 **Data-Centric primero**: antes de probar 50 modelos, mejora la calidad de tus datos.

## <span style="color: #2826DE;">Actividad: Exposición por Equipos (15 mins)

Cada equipo preparará una mini-presentación de 3 minutos sobre un subtema asignado. El objetivo es que practiquen explicar conceptos técnicos de forma clara y concisa.

**Temas asignados:**

| Equipo | Tema | Pregunta que deben responder |
|---|---|---|
Equipo 1 | Bootstrapping | ¿Cómo nos ayuda a medir la incertidumbre de un modelo? |
Equipo 2 | Tipos de Data Leakage | ¿Cuáles son las 3 formas más comunes y cómo detectarlas? |
Equipo 3 | Beneficios de Pipeline | ¿Por qué usar Pipeline en vez de preprocesar manualmente? |
Equipo 4 | Data-Centric AI | ¿Por qué mejorar datos puede ser mejor que mejorar modelos? |

**Formato:**
- 3 minutos de presentación por equipo
- Pueden usar pizarrón, compartir pantalla, o solo explicar verbalmente
- Al final, 1 pregunta del público para cada equipo

**Evaluación informal:** Claridad de la explicación + uso de ejemplos concretos.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué es bootstrapping?

- Un método para limpiar datos
- Muestreo con reemplazo para estimar distribuciones de estadísticas 
- Una técnica de regularización
- Un tipo de cross-validation


2️⃣ ¿Cuál es un ejemplo de target leakage?

- Usar datos del futuro en series temporales
- Usar una variable que se genera DESPUÉS del evento a predecir 
- Hacer fit del scaler en todo el dataset
- No usar cross-validation


3️⃣ ¿Qué previene automáticamente un sklearn Pipeline?

- Overfitting
- Train-test contamination en preprocesamiento 
- Que el modelo sea muy lento
- Que haya valores nulos


4️⃣ ¿Qué red flag indica posible data leakage?

- Accuracy baja en train y test
- El modelo tarda mucho en entrenar
- Rendimiento sospechosamente alto que cae en producción 
- Features con muchos valores nulos


5️⃣ ¿Qué propone el enfoque Data-Centric AI?

- Usar modelos más complejos
- Enfocarse en mejorar la calidad de los datos en vez del modelo 
- Eliminar todos los features categóricos
- Usar solo deep learning


6️⃣ ¿Cuál es la ventaja de reportar un intervalo de confianza en vez de solo accuracy?

- Es más rápido de calcular
- Comunica la incertidumbre real de la estimación 
- Siempre da valores más altos
- Es requerido por sklearn